In [2]:
# ============================================================
# Full ML Pipeline Assessment: Palmer Penguins
# From Clustering to Deployment
# ============================================================

import warnings
warnings.filterwarnings("ignore")

import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from IPython.display import display, Markdown

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate,
    GridSearchCV,
    learning_curve
)

from sklearn.preprocessing import StandardScaler, OneHotEncoder, label_binarize
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans, DBSCAN

from sklearn.metrics import (
    silhouette_score,
    adjusted_rand_score,
    normalized_mutual_info_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    ConfusionMatrixDisplay,
    roc_curve,
    auc
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.inspection import permutation_importance


# ============================================================
# Helper Functions
# ============================================================

def section(title):
    display(Markdown(f"\n# {title}\n"))


def subsection(title):
    display(Markdown(f"\n## {title}\n"))


def make_onehot_encoder():
    """
    Handles different scikit-learn versions.
    New versions use sparse_output=False.
    Older versions use sparse=False.
    """
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def safe_silhouette(X, labels):
    """
    Silhouette score is valid only when there are at least 2 clusters.
    """
    unique_labels = np.unique(labels)
    if len(unique_labels) < 2 or len(unique_labels) >= len(labels):
        return np.nan
    return silhouette_score(X, labels)


# ============================================================
# Task 1 — Unsupervised Exploration
# ============================================================

section("Task 1 — Unsupervised Exploration")

# Load dataset
df = sns.load_dataset("penguins")

subsection("Dataset Overview")
display(df.head())
display(df.info())
display(Markdown(f"Original dataset shape: **{df.shape}**"))

subsection("Missing Values")
missing_values = df.isna().sum().sort_values(ascending=False)
display(missing_values)

# Clean data
df_clean = df.dropna().copy()

display(Markdown(f"""
After dropping missing values, the dataset shape is **{df_clean.shape}**.

Missing values were removed because the dataset is small and the assessment asks for a complete ML workflow.
For this dataset, dropping rows with missing values is acceptable because only a small number of rows are affected.
"""))

subsection("Distribution Check")

numeric_features = df_clean.select_dtypes(include=["number"]).columns.tolist()

for col in numeric_features:
    plt.figure(figsize=(7, 4))
    sns.histplot(df_clean[col], kde=True)
    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Count")
    plt.show()

display(Markdown("""
### Notes on distributions and anomalies

The numeric variables show natural biological variation across penguin species.
There are no extreme impossible values after removing missing rows.
Some distributions appear multi-modal, which is expected because the dataset contains several species with different body sizes and bill measurements.
"""))

# Scale numeric features
X_numeric = df_clean[numeric_features].copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_numeric)

# PCA
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

# t-SNE
tsne = TSNE(
    n_components=2,
    random_state=42,
    perplexity=30,
    init="pca",
    learning_rate="auto"
)
X_tsne = tsne.fit_transform(X_scaled)

# Side-by-side plot colored by actual species
subsection("PCA and t-SNE Colored by Actual Species")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.scatterplot(
    x=X_pca[:, 0],
    y=X_pca[:, 1],
    hue=df_clean["species"],
    ax=axes[0],
    palette="deep"
)
axes[0].set_title("PCA Projection Colored by Actual Species")
axes[0].set_xlabel("PC1")
axes[0].set_ylabel("PC2")

sns.scatterplot(
    x=X_tsne[:, 0],
    y=X_tsne[:, 1],
    hue=df_clean["species"],
    ax=axes[1],
    palette="deep"
)
axes[1].set_title("t-SNE Projection Colored by Actual Species")
axes[1].set_xlabel("t-SNE 1")
axes[1].set_ylabel("t-SNE 2")

plt.tight_layout()
plt.show()

display(Markdown(f"""
PCA explained variance ratio:

- PC1: **{pca.explained_variance_ratio_[0]:.3f}**
- PC2: **{pca.explained_variance_ratio_[1]:.3f}**
- Total: **{pca.explained_variance_ratio_.sum():.3f}**
"""))

# Clustering
subsection("K-Means and DBSCAN Clustering")

species_labels = df_clean["species"].astype("category").cat.codes

clustering_results = []

# K-Means k=3
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(X_scaled)

clustering_results.append({
    "method": "K-Means",
    "params": "k=3",
    "labels": kmeans_labels,
    "silhouette": safe_silhouette(X_scaled, kmeans_labels),
    "ARI": adjusted_rand_score(species_labels, kmeans_labels),
    "NMI": normalized_mutual_info_score(species_labels, kmeans_labels)
})

# DBSCAN experiments: at least 2 combinations
dbscan_configs = [
    {"eps": 0.7, "min_samples": 5},
    {"eps": 1.0, "min_samples": 5},
    {"eps": 1.2, "min_samples": 5},
]

for config in dbscan_configs:
    dbscan = DBSCAN(eps=config["eps"], min_samples=config["min_samples"])
    dbscan_labels = dbscan.fit_predict(X_scaled)

    clustering_results.append({
        "method": "DBSCAN",
        "params": f"eps={config['eps']}, min_samples={config['min_samples']}",
        "labels": dbscan_labels,
        "silhouette": safe_silhouette(X_scaled, dbscan_labels),
        "ARI": adjusted_rand_score(species_labels, dbscan_labels),
        "NMI": normalized_mutual_info_score(species_labels, dbscan_labels)
    })

clustering_summary = pd.DataFrame([
    {
        "method": result["method"],
        "params": result["params"],
        "silhouette": result["silhouette"],
        "ARI": result["ARI"],
        "NMI": result["NMI"],
        "n_clusters_found": len(set(result["labels"])),
        "noise_points": int(np.sum(result["labels"] == -1))
    }
    for result in clustering_results
])

display(clustering_summary)

# Select best clustering by silhouette
valid_results = [r for r in clustering_results if not np.isnan(r["silhouette"])]

if valid_results:
    best_clustering = max(valid_results, key=lambda x: x["silhouette"])
else:
    best_clustering = clustering_results[0]

best_labels = best_clustering["labels"]

subsection("Best Clustering Visualized on PCA Projection")

plt.figure(figsize=(8, 6))
sns.scatterplot(
    x=X_pca[:, 0],
    y=X_pca[:, 1],
    hue=best_labels,
    palette="deep"
)
plt.title(f"Best Clustering on PCA Projection: {best_clustering['method']} ({best_clustering['params']})")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend(title="Cluster")
plt.show()

display(Markdown(f"""
### Task 1 Interpretation

The best clustering method by silhouette score was:

**{best_clustering['method']} ({best_clustering['params']})**

Results:

- Silhouette score: **{best_clustering['silhouette']:.3f}**
- Adjusted Rand Index: **{best_clustering['ARI']:.3f}**
- Normalized Mutual Information: **{best_clustering['NMI']:.3f}**

Unsupervised methods recovered part of the penguin species structure because species differ in body mass, flipper length, and bill measurements.
However, clustering is not perfect because biological measurements can overlap between species.
K-Means works better when clusters are compact and roughly spherical.
DBSCAN can struggle when clusters have different densities or when the correct `eps` value is hard to tune.
""")


# ============================================================
# Task 2 — Supervised Model Pipeline
# ============================================================

section("Task 2 — Supervised Model Pipeline")

# Prepare full dataset
df_supervised = df.dropna().copy()

target = "species"
X = df_supervised.drop(columns=[target])
y = df_supervised[target]

numeric_cols = X.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()

numeric_cols = X.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()

print("Target variable:", target)
print("Numeric features:", numeric_cols)
print("Categorical features:", categorical_cols)

# Preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", make_onehot_encoder(), categorical_cols)
    ]
)

# Candidate models
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "SVC": SVC(probability=True, random_state=42)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    "accuracy": "accuracy",
    "precision_macro": "precision_macro",
    "recall_macro": "recall_macro",
    "f1_macro": "f1_macro"
}

cv_results = []

for model_name, model in models.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    scores = cross_validate(
        pipeline,
        X,
        y,
        cv=cv,
        scoring=scoring,
        return_train_score=False
    )

    cv_results.append({
        "model": model_name,
        "accuracy": scores["test_accuracy"].mean(),
        "precision_macro": scores["test_precision_macro"].mean(),
        "recall_macro": scores["test_recall_macro"].mean(),
        "f1_macro": scores["test_f1_macro"].mean()
    })

cv_results_df = pd.DataFrame(cv_results).sort_values("f1_macro", ascending=False)
display(cv_results_df)

best_model_name = cv_results_df.iloc[0]["model"]

display(Markdown(f"""
The best default model based on macro F1 score is:

**{best_model_name}**
"""))

# Build selected default pipeline
selected_default_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", models[best_model_name])
])

# Hyperparameter grid with at least 3 parameters
if best_model_name == "Logistic Regression":
    param_grid = {
        "model__C": [0.01, 0.1, 1, 10],
        "model__solver": ["lbfgs", "liblinear"],
        "model__penalty": ["l2"]
    }

elif best_model_name == "Random Forest":
    param_grid = {
        "model__n_estimators": [100, 200, 300],
        "model__max_depth": [None, 3, 5, 10],
        "model__min_samples_split": [2, 5, 10]
    }

else:
    param_grid = {
        "model__C": [0.1, 1, 10],
        "model__kernel": ["linear", "rbf"],
        "model__gamma": ["scale", "auto"]
    }

grid_search = GridSearchCV(
    estimator=selected_default_pipeline,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1
)

grid_search.fit(X, y)

best_model = grid_search.best_estimator_

display(Markdown("## GridSearchCV Results"))

print("Best model:", best_model_name)
print("Best parameters:", grid_search.best_params_)
print("Best CV macro F1:", grid_search.best_score_)

default_f1 = cv_results_df.loc[cv_results_df["model"] == best_model_name, "f1_macro"].iloc[0]

display(Markdown(f"""
### Default vs Tuned Model

Default **{best_model_name}** macro F1:

**{default_f1:.4f}**

Tuned **{best_model_name}** macro F1:

**{grid_search.best_score_:.4f}**

The tuned model is selected as the final model for evaluation and deployment.
"""))


# ============================================================
# Task 3 — Model Evaluation & Interpretation
# ============================================================

section("Task 3 — Model Evaluation & Interpretation")

# Held-out train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)

subsection("Classification Report")
print(classification_report(y_test, y_pred))

# Confusion matrix
subsection("Confusion Matrix")

fig, ax = plt.subplots(figsize=(7, 5))
ConfusionMatrixDisplay.from_estimator(
    best_model,
    X_test,
    y_test,
    ax=ax,
    cmap="Blues"
)
plt.title("Confusion Matrix")
plt.show()

# ROC curves one-vs-rest
subsection("ROC Curves One-vs-Rest")

classes = best_model.classes_
y_test_binarized = label_binarize(y_test, classes=classes)

if hasattr(best_model, "predict_proba"):
    y_score = best_model.predict_proba(X_test)
else:
    raise ValueError("The selected model does not support predict_proba.")

plt.figure(figsize=(8, 6))

roc_auc_values = {}

for i, class_name in enumerate(classes):
    fpr, tpr, _ = roc_curve(y_test_binarized[:, i], y_score[:, i])
    roc_auc = auc(fpr, tpr)
    roc_auc_values[class_name] = roc_auc

    plt.plot(fpr, tpr, label=f"{class_name} AUC = {roc_auc:.3f}")

plt.plot([0, 1], [0, 1], linestyle="--")
plt.title("One-vs-Rest ROC Curves")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.show()

display(pd.DataFrame({
    "species": list(roc_auc_values.keys()),
    "AUC": list(roc_auc_values.values())
}))

# Learning curves
subsection("Learning Curves")

train_sizes, train_scores, validation_scores = learning_curve(
    estimator=best_model,
    X=X,
    y=y,
    train_sizes=np.linspace(0.1, 1.0, 10),
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1
)

train_scores_mean = train_scores.mean(axis=1)
validation_scores_mean = validation_scores.mean(axis=1)

plt.figure(figsize=(8, 6))
plt.plot(train_sizes, train_scores_mean, marker="o", label="Training score")
plt.plot(train_sizes, validation_scores_mean, marker="o", label="Validation score")
plt.title("Learning Curve")
plt.xlabel("Training Set Size")
plt.ylabel("Macro F1 Score")
plt.legend()
plt.grid(True)
plt.show()

# Permutation importance
subsection("Permutation Importances")

perm_importance = permutation_importance(
    best_model,
    X_test,
    y_test,
    scoring="f1_macro",
    n_repeats=20,
    random_state=42,
    n_jobs=-1
)

importance_df = pd.DataFrame({
    "feature": X_test.columns,
    "importance_mean": perm_importance.importances_mean,
    "importance_std": perm_importance.importances_std
}).sort_values("importance_mean", ascending=False)

display(importance_df)

plt.figure(figsize=(8, 5))
sns.barplot(
    data=importance_df,
    x="importance_mean",
    y="feature"
)
plt.title("Permutation Importances on Test Set")
plt.xlabel("Mean Importance")
plt.ylabel("Feature")
plt.show()

# Hardest species
report_dict = classification_report(y_test, y_pred, output_dict=True)
species_f1 = {
    species: report_dict[species]["f1-score"]
    for species in classes
}
hardest_species = min(species_f1, key=species_f1.get)

top_features = importance_df.head(3)["feature"].tolist()

train_val_gap = train_scores_mean[-1] - validation_scores_mean[-1]

if train_val_gap > 0.10:
    fitting_status = "There may be mild overfitting because the training score is noticeably higher than the validation score."
elif validation_scores_mean[-1] < 0.80:
    fitting_status = "There may be underfitting because both training and validation scores are not very high."
else:
    fitting_status = "There is no strong sign of severe overfitting or underfitting. Training and validation performance are both strong."

display(Markdown(f"""
## Comprehensive Interpretation

### Is the model overfitting or underfitting?

{fitting_status}

The final training macro F1 score from the learning curve is approximately **{train_scores_mean[-1]:.3f}**.
The final validation macro F1 score from the learning curve is approximately **{validation_scores_mean[-1]:.3f}**.

### Which species is hardest to classify and why?

The hardest species to classify in the held-out test set appears to be:

**{hardest_species}**

This is based on the lowest F1 score in the classification report.
Classification difficulty can happen when one species overlaps with another species in bill size, flipper length, or body mass.

### Which features drive predictions the most?

The strongest features according to permutation importance are:

**{top_features}**

These features make biological sense because penguin species differ strongly in body size, flipper length, and bill measurements.

### Are there any signs of data leakage or evaluation issues?

There is no obvious data leakage because the target variable `species` was removed from the feature matrix before training.
The preprocessing was placed inside a scikit-learn Pipeline, which means scaling and one-hot encoding are fitted only inside cross-validation folds or training data.
This avoids leaking information from validation/test data into preprocessing.

One limitation is that the Palmer Penguins dataset is small.
Because of this, performance estimates can be sensitive to the exact train/test split.
""" ))


# ============================================================
# Task 4 — Model Deployment Prototype
# ============================================================

section("Task 4 — Model Deployment Prototype")

# Save model artifact
MODEL_PATH = "penguin_species_pipeline.joblib"
joblib.dump(best_model, MODEL_PATH)

display(Markdown(f"""
The final tuned pipeline was serialized with joblib.

Saved model artifact:

`{MODEL_PATH}`
"""))

# Create app.py automatically
app_code = r'''
from flask import Flask, request, jsonify
import joblib
import pandas as pd
import os

app = Flask(__name__)

MODEL_PATH = "penguin_species_pipeline.joblib"

REQUIRED_FIELDS = [
    "island",
    "bill_length_mm",
    "bill_depth_mm",
    "flipper_length_mm",
    "body_mass_g",
    "sex",
]

NUMERIC_FIELDS = [
    "bill_length_mm",
    "bill_depth_mm",
    "flipper_length_mm",
    "body_mass_g",
]

CATEGORICAL_FIELDS = [
    "island",
    "sex",
]


def load_model():
    if not os.path.exists(MODEL_PATH):
        raise FileNotFoundError(
            f"Model file not found: {MODEL_PATH}. "
            "Make sure the .joblib file is in the same folder as app.py."
        )
    return joblib.load(MODEL_PATH)


model = load_model()


@app.route("/health", methods=["GET"])
def health():
    return jsonify({
        "status": "ok",
        "message": "Penguin species prediction API is running"
    }), 200


def validate_input(data):
    if not isinstance(data, dict):
        return False, "Input must be a JSON object."

    missing_fields = [field for field in REQUIRED_FIELDS if field not in data]
    if missing_fields:
        return False, f"Missing required fields: {missing_fields}"

    for field in NUMERIC_FIELDS:
        try:
            float(data[field])
        except (ValueError, TypeError):
            return False, f"Field '{field}' must be numeric."

    for field in CATEGORICAL_FIELDS:
        if data[field] is None or str(data[field]).strip() == "":
            return False, f"Field '{field}' must be a non-empty string."

    return True, None


@app.route("/predict", methods=["POST"])
def predict():
    try:
        data = request.get_json()

        is_valid, error_message = validate_input(data)
        if not is_valid:
            return jsonify({
                "error": error_message
            }), 400

        input_df = pd.DataFrame([{
            "island": data["island"],
            "bill_length_mm": float(data["bill_length_mm"]),
            "bill_depth_mm": float(data["bill_depth_mm"]),
            "flipper_length_mm": float(data["flipper_length_mm"]),
            "body_mass_g": float(data["body_mass_g"]),
            "sex": data["sex"],
        }])

        prediction = model.predict(input_df)[0]

        response = {
            "predicted_species": prediction
        }

        if hasattr(model, "predict_proba"):
            probabilities = model.predict_proba(input_df)[0]

            if hasattr(model, "classes_"):
                class_names = model.classes_
            elif hasattr(model, "named_steps") and hasattr(model.named_steps.get("model"), "classes_"):
                class_names = model.named_steps["model"].classes_
            else:
                class_names = [f"class_{i}" for i in range(len(probabilities))]

            response["class_probabilities"] = {
                str(class_name): float(prob)
                for class_name, prob in zip(class_names, probabilities)
            }

        return jsonify(response), 200

    except Exception as e:
        return jsonify({
            "error": "Prediction failed.",
            "details": str(e)
        }), 500


if __name__ == "__main__":
    app.run(debug=True)
'''

with open("app.py", "w", encoding="utf-8") as f:
    f.write(app_code)

display(Markdown("Created `app.py` successfully."))

# Test API using Flask test client from app.py
subsection("API Test From Notebook")

import importlib.util
import sys

spec = importlib.util.spec_from_file_location("penguin_app", "app.py")
penguin_app = importlib.util.module_from_spec(spec)
sys.modules["penguin_app"] = penguin_app
spec.loader.exec_module(penguin_app)

client = penguin_app.app.test_client()

# Health endpoint test
health_response = client.get("/health")

print("Health endpoint status code:", health_response.status_code)
print("Health endpoint response:")
print(json.dumps(health_response.get_json(), indent=2))

# Valid request
sample_valid_input = X_test.iloc[0].to_dict()

valid_response = client.post(
    "/predict",
    json=sample_valid_input
)

print("\nValid /predict status code:", valid_response.status_code)
print("Valid /predict response:")
print(json.dumps(valid_response.get_json(), indent=2))

# Invalid request
sample_invalid_input = {
    "island": "Torgersen",
    "bill_length_mm": "not_a_number",
    "bill_depth_mm": 18.7,
    "flipper_length_mm": 181,
    "body_mass_g": 3750,
    "sex": "Male"
}

invalid_response = client.post(
    "/predict",
    json=sample_invalid_input
)

print("\nInvalid /predict status code:", invalid_response.status_code)
print("Invalid /predict response:")
print(json.dumps(invalid_response.get_json(), indent=2))


SyntaxError: '(' was never closed (3214527906.py, line 275)